# Segmentation Prediction Model (KMeans)

This notebook trains a KMeans clustering model to dynamically assign users to macro-segments based on RFM and category affinity data.

In [ ]:
import os
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_theme(style="darkgrid")

## 1. Load Data & Explore

In [ ]:
data_path = os.path.join("..", "processed", "segmentation_ready.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
display(df.head())

## 2. Preprocessing & Scaling

In [ ]:
features = ['avg_order_value', 'purchase_frequency', 'recency_days', 'category_affinity_score']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"Scaled features shape: {X_scaled.shape}")

## 3. Elbow Method Verification

In [ ]:
inertia = []
K_range = range(1, 9)
for k in K_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42, n_init='auto')
    kmeans_temp.fit(X_scaled)
    inertia.append(kmeans_temp.inertia_)

plt.figure(figsize=(6, 4))
plt.plot(K_range, inertia, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.show()

## 4. Train KMeans Classifier

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init='auto')
cluster_labels = kmeans.fit_predict(X_scaled)
df['cluster'] = cluster_labels

## 5. Evaluation & Centroids Extraction

In [ ]:
centroids = pd.DataFrame(scaler.inverse_transform(kmeans.cluster_centers_), columns=features)
print("Cluster Centroids:")
display(centroids)

In [ ]:
labels_map = {
    0: 'loyal_premium',
    1: 'first_timer',
    2: 'discount_seeker',
    3: 'hesitant_buyer'
}
df['business_label'] = df['cluster'].map(labels_map)

print("Segment Distribution:")
print(df['business_label'].value_counts())

## 6. Cluster Visualization (PCA)

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='business_label', data=df, palette='Set2')
plt.title('Customer Segments PCA Visualization')
plt.show()

## 7. Export Model Bundle

In [ ]:
models_dir = os.path.join("..", "..", "backend", "models")
os.makedirs(models_dir, exist_ok=True)
model_path = os.path.join(models_dir, "segmentation.pkl")

bundle = {
    'model': kmeans,
    'scaler': scaler,
    'labels_map': labels_map,
    'features': features
}
joblib.dump(bundle, model_path)
print(f"Saved Model successfully to {model_path}")